# Module 1 - Topic 5: Structured Outputs & Validation
**AI Application Development - Developer Track (NIGCOMSAT)**

In this demo you turn a plain-text NOC incident report into a validated, structured Python object -- the exact skill the Week 1 deliverable needs. One concept per cell; run them in order.

## Install
Run this once if these packages are not already installed. Remove the leading `#` to run, then add it back.

In [ ]:
# !pip install openai python-dotenv pydantic

## Import & Setup
We load our credentials from `.env` (see `.env.example` at the repository root) and create the API client.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)
MODEL_NAME = os.getenv("MODEL_NAME")

print("Client ready. Model:", MODEL_NAME)

Client ready. Model: google/gemma-4-E4B-it


## The Source Document
This is the synthetic NOC incident report from `shared/synthetic-mce-corpus/NOC-INC-2026-0417_noc_incident_report.txt`. It is pasted in full here so the notebook runs without needing a file path. No real NIGCOMSAT data appears anywhere in this document -- every name, reference number and company is invented.

In [2]:
INCIDENT_REPORT = """*** SYNTHETIC DOCUMENT - FOR TRAINING USE ONLY - NOT A REAL NIGCOMSAT RECORD ***

NETWORK OPERATIONS CENTRE (NOC)
INCIDENT REPORT

Incident ID:            NOC-INC-2026-0417
Report status:          Final
Prepared by:            T. Okafor, Shift Engineer (Night Shift B)
Reviewed by:            M. Danjuma, NOC Duty Manager
Report date:            2026-04-17

------------------------------------------------------------
1. SUMMARY
------------------------------------------------------------
Loss of carrier on the Ku-band link between Hub 1 and Gateway 3
caused a service outage for three enterprise customers in the early
hours of 17 April 2026. Service was restored after the failed
uplink power amplifier was bypassed and traffic was moved to the
backup carrier. The outage lasted approximately two hours.

------------------------------------------------------------
2. INCIDENT DETAILS
------------------------------------------------------------
Affected link:          KU-HUB1-GW3
Service type:           Ku-band enterprise VSAT backhaul
Severity:               P2 - High
Customers affected:     3 (Harmattan Telecom Ltd, Savanna Rural
                        Connect, Rivers Basin Logistics)
Terminals affected:     41

------------------------------------------------------------
3. TIMELINE (all times WAT, 24-hour clock)
------------------------------------------------------------
2026-04-17 02:14   Loss-of-carrier alarm raised on KU-HUB1-GW3.
                   Detected by the monitoring system (alarm ref
                   ALM-88231).
2026-04-17 02:31   Escalated by shift engineer to the RF Systems
                   on-call engineer and the NOC Duty Manager.
2026-04-17 03:05   Mitigation applied: traffic moved to backup
                   carrier KU-HUB1-GW3-B. Backup capacity is 60
                   percent of the primary, so service was partly
                   restored with reduced throughput.
2026-04-17 03:52   Full service restored after the uplink power
                   amplifier (HPA-2) was bypassed and the primary
                   carrier returned to normal levels.

------------------------------------------------------------
4. ROOT CAUSE
------------------------------------------------------------
Hardware fault in uplink power amplifier HPA-2 at Hub 1, causing
loss of the transmitted carrier. The unit was found to have a failed
cooling fan and had shut down on over-temperature. HPA-2 was
bypassed pending replacement.

------------------------------------------------------------
5. ACTIONS TAKEN
------------------------------------------------------------
- Alarm acknowledged and triaged by the night shift engineer.
- RF Systems on-call engineer confirmed the amplifier fault by
  remote inspection.
- Traffic moved to backup carrier to reduce customer impact.
- HPA-2 bypassed and primary carrier restored.
- The three affected customers were notified by the service desk at
  03:20 and again at 04:00 when full service was confirmed.

------------------------------------------------------------
6. FOLLOW-UP
------------------------------------------------------------
- Replace HPA-2 cooling fan and return HPA-2 to service.
  Owner: RF Systems. Target date: 2026-04-24.
- Post-incident review to be held by 2026-04-24. Owner: NOC Duty
  Manager.
- Review over-temperature alarm thresholds on all Hub 1 amplifiers.
  Owner: RF Systems. Target date: 2026-05-08.
"""

print(INCIDENT_REPORT[:300], "...")

*** SYNTHETIC DOCUMENT - FOR TRAINING USE ONLY - NOT A REAL NIGCOMSAT RECORD ***

NETWORK OPERATIONS CENTRE (NOC)
INCIDENT REPORT

Incident ID:            NOC-INC-2026-0417
Report status:          Final
Prepared by:            T. Okafor, Shift Engineer (Night Shift B)
Reviewed by:            M. Danj ...


## Defining the Contract
One Pydantic class does two jobs: it generates the JSON Schema we send to the API, and it validates the reply once it comes back. We also add a business rule -- `customers_affected` cannot be negative -- that the schema's `int` type alone cannot express.

In [3]:
from pydantic import BaseModel, field_validator
from typing import Literal, Optional


class IncidentExtract(BaseModel):
    incident_id: str
    severity: Literal["P1", "P2", "P3", "P4"]
    affected_link: str
    customers_affected: int
    root_cause: Optional[str] = None  # None if the report does not state a cause

    @field_validator("customers_affected")
    @classmethod
    def must_be_non_negative(cls, v):
        if v < 0:
            raise ValueError("customers_affected cannot be negative")
        return v


IncidentExtract.model_json_schema()

{'properties': {'incident_id': {'title': 'Incident Id', 'type': 'string'},
  'severity': {'enum': ['P1', 'P2', 'P3', 'P4'],
   'title': 'Severity',
   'type': 'string'},
  'affected_link': {'title': 'Affected Link', 'type': 'string'},
  'customers_affected': {'title': 'Customers Affected', 'type': 'integer'},
  'root_cause': {'anyOf': [{'type': 'string'}, {'type': 'null'}],
   'default': None,
   'title': 'Root Cause'}},
 'required': ['incident_id',
  'severity',
  'affected_link',
  'customers_affected'],
 'title': 'IncidentExtract',
 'type': 'object'}

## Making a Structured Output Call
We pass `IncidentExtract` directly as `response_format`. The API constrains its reply to this shape, and the SDK parses it straight into an `IncidentExtract` instance -- no manual `json.loads()` needed.

In [4]:
response = client.chat.completions.parse(
    model=MODEL_NAME,
    messages=[
        {
            "role": "system",
            "content": (
                "Extract incident details from the NOC report below. "
                "If the report does not state a root cause, leave root_cause as null "
                "rather than guessing."
            ),
        },
        {"role": "user", "content": INCIDENT_REPORT},
    ],
    response_format=IncidentExtract,
)

incident = response.choices[0].message.parsed
print(type(incident))
print(incident.model_dump_json(indent=2))

<class '__main__.IncidentExtract'>
{
  "incident_id": "NOC-INC-2026-0417",
  "severity": "P2",
  "affected_link": "KU-HUB1-GW3",
  "customers_affected": 3,
  "root_cause": "Hardware fault in uplink power amplifier HPA-2 at Hub 1 (failed cooling fan causing over-temperature shutdown)"
}


## Validation Catches What the Schema Can't
Structured output mode guarantees `customers_affected` is an `int`. It says nothing about whether that `int` makes sense. Here we simulate a reply where the model returned a nonsensical value, and let our business validator catch it.

In [5]:
from pydantic import ValidationError

bad_reply = {
    "incident_id": "NOC-INC-2026-0417",
    "severity": "P2",
    "affected_link": "KU-HUB1-GW3",
    "customers_affected": -3,   # nonsense, but schema-legal -- it is still an int
    "root_cause": "unknown",
}

try:
    IncidentExtract(**bad_reply)
except ValidationError as e:
    print(e)

1 validation error for IncidentExtract
customers_affected
  Value error, customers_affected cannot be negative [type=value_error, input_value=-3, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/value_error


## The Retry Loop
When validation fails, the cleanest fix is to tell the model exactly what was wrong and ask again, capped at a small number of attempts. A model that keeps failing needs a human, not a fourth try.

In [6]:
def extract_with_retry(report_text: str, max_attempts: int = 3) -> IncidentExtract:
    messages = [
        {
            "role": "system",
            "content": (
                "Extract incident details from the NOC report below. "
                "If the report does not state a root cause, leave root_cause as null "
                "rather than guessing."
            ),
        },
        {"role": "user", "content": report_text},
    ]

    for attempt in range(1, max_attempts + 1):
        try:
            response = client.chat.completions.parse(
                model=MODEL_NAME,
                messages=messages,
                response_format=IncidentExtract,
            )
            return response.choices[0].message.parsed
        except ValidationError as e:
            print(f"Attempt {attempt} failed validation:\n{e}\n")
            messages.append({
                "role": "user",
                "content": f"That did not validate: {e}. Please correct it and try again.",
            })

    raise RuntimeError(f"Could not get valid structured output after {max_attempts} attempts")


incident = extract_with_retry(INCIDENT_REPORT)
print("Succeeded:")
print(incident.model_dump_json(indent=2))

Succeeded:
{
  "incident_id": "NOC-INC-2026-0417",
  "severity": "P2",
  "affected_link": "KU-HUB1-GW3",
  "customers_affected": 3,
  "root_cause": "Hardware fault in uplink power amplifier HPA-2 at Hub 1 (failed cooling fan causing over-temperature shutdown)."
}


## Optional Fields: Letting the Model Say "Not in the Document"
Not every report states a root cause -- this one is still under investigation. Because `root_cause` is `Optional`, the model can leave it `null` instead of inventing one.

In [7]:
SHORT_REPORT = """*** SYNTHETIC DOCUMENT - FOR TRAINING USE ONLY - NOT A REAL NIGCOMSAT RECORD ***

NETWORK OPERATIONS CENTRE (NOC)
INCIDENT REPORT (PRELIMINARY)

Incident ID:         NOC-INC-2026-0501
Severity:            P3 - Medium
Affected link:       KU-HUB2-GW1
Customers affected:  1

Summary: Brief loss of signal on KU-HUB2-GW1 lasting under five
minutes. Service recovered on its own before the on-call engineer
could complete diagnostics. Investigation into the cause is ongoing;
no cause has been confirmed yet. This is a preliminary report and
will be updated once the investigation concludes.
"""

incident_2 = extract_with_retry(SHORT_REPORT)
print(incident_2.model_dump_json(indent=2))
assert incident_2.root_cause is None, "Expected root_cause to be None when the report does not state a cause"

{
  "incident_id": "NOC-INC-2026-0501",
  "severity": "P3",
  "affected_link": "KU-HUB2-GW1",
  "customers_affected": 1,
  "root_cause": null
}


## Recap & What's Next
- `IncidentExtract` is now a working, validated extractor for NOC incident reports
- Structured output mode gave us the shape guarantee; a `field_validator` gave us the business rule
- The retry loop is the safety net for when the schema is satisfied but validation still fails
- Optional fields let the model admit "not stated" instead of guessing

Topic 6 turns this same extractor into something you can actually trust: a test set with known answers -- including the "two hours" versus 1h38m discrepancy hiding in this report's own summary -- and the first `pytest` tests that prove the validation works.